# Amazon Bedrock Prompt Caching with Converse API

In this notebook, you will explore the **Amazon Bedrock Converse API** in **us-east-2 (Ohio)** and learn how to use **Prompt Caching** (`cachePoint`) to:
- Cache static or repetitive context (like system instructions and few-shot examples).
- Reduce Time-to-First-Token (TTFT) latency.
- Conserve tokens and reduce the overall cost of using Bedrock.
- Understand the difference between **Cache Write** (initial request) and **Cache Read** (subsequent requests).

---

## Prerequisites
Before you start, make sure you have:
1. **Python 3.x** installed.
2. **AWS CLI** configured (`aws configure`) with valid AWS credentials.
3. Access to **Amazon Nova Micro** in region **`us-east-2`** via its cross-region inference profile (`us.amazon.nova-micro-v1:0`).
4. Required Python packages: `boto3`, `jupyter`.


## Task 1: Creating an Exercise Environment

To begin, you can create a dedicated Python virtual environment and install the required packages:

**Windows (PowerShell):**
```powershell
cd ~/Desktop
python -m venv exercise7
.\exercise7\Scripts\Activate.ps1
pip install boto3 jupyter
```

**macOS / Linux:**
```bash
cd ~/Desktop
python3 -m venv exercise7
source exercise7/bin/activate
pip install boto3 jupyter
```


## Task 2: Caching Bedrock Responses

In this assignment, you will explore the Amazon Bedrock Converse API through an interactive chat session. You will learn how to cache responses and conserve tokens to reduce the overall cost of using Bedrock.

### Task 2.1: Setting up Bedrock (Region: `us-east-2`)
Initialize the Boto3 client for `bedrock-runtime` explicitly in **`us-east-2`** and set up the model identifier.

> [!NOTE]
> Amazon Nova foundation models in Bedrock require the use of a cross-region inference profile ID (prefixed with `us.`, such as `us.amazon.nova-micro-v1:0`).


In [1]:
import boto3
import json
from datetime import datetime

# Explicitly initialize Bedrock Runtime client in us-east-2
REGION = "us-east-2"
bedrock = boto3.client('bedrock-runtime', region_name=REGION)  

# Foundation model identifier (Cross-Region System Inference Profile)
MODEL_ID = "us.amazon.nova-micro-v1:0"

print(f"Bedrock Runtime client initialized for region '{REGION}' with model: {MODEL_ID}")


Bedrock Runtime client initialized for region 'us-east-2' with model: us.amazon.nova-micro-v1:0


### Task 2.2: Training Bedrock with a Few-Shot System Prompt

This tells Bedrock what it will be doing. It provides examples of customer reviews and their desired summaries to guide model behavior.


In [2]:
system_prompt = """
You are an assistant that summarizes music reviews for a record company.
Here are examples:

Review: The latest album by The New Wave Band is a masterpiece! Every track is a hit.
Summary: Reviewer praises the latest album as a masterpiece with hit tracks.

Review: I was disappointed with the new single; it lacked the energy of their previous work.
Summary: Reviewer expresses disappointment, noting a lack of energy compared to previous work.
"""

# Input review to summarize
user_input_review = "Review: The new EP is a solid effort with a few standout songs, but some tracks feel repetitive."

print("System prompt and user review defined successfully.")


System prompt and user review defined successfully.


### Task 2.3: No Caching (Baseline)

Run the request without caching. Notice the total token usage as this number will be compared to the cached response in the next step.


In [3]:
no_cache_payload = {
    "system": [
        {"text": system_prompt}
    ],
    "messages": [
        {
            "role": "user",
            "content": [
                {"text": user_input_review + "\nSummary:"}
            ]
        }
    ]
}

response = bedrock.converse(
    modelId=MODEL_ID,
    **no_cache_payload
)

output_text = response['output']['message']['content'][0]['text']
usage_no_cache = response['usage']
total_tokens_no_cache = usage_no_cache['totalTokens']

print("[No Caching] Generated Summary:")
print(output_text)
print(f"\nToken Usage Breakdown (No Cache):")
print(f"  Input Tokens:  {usage_no_cache['inputTokens']}")
print(f"  Output Tokens: {usage_no_cache['outputTokens']}")
print(f"  Total Tokens:  {total_tokens_no_cache}")


[No Caching] Generated Summary:
Reviewer acknowledges the new EP as a solid effort with standout tracks, although some tracks are noted to be repetitive.

Token Usage Breakdown (No Cache):
  Input Tokens:  109
  Output Tokens: 23
  Total Tokens:  132


> **Expected Baseline**:
> - `inputTokens`: ~109 (87 system prompt tokens + 22 user review tokens)
> - `outputTokens`: ~23 (model summary)
> - `totalTokens`: ~132
> 
> Since no caching was enabled, all 87 tokens of the system prompt had to be parsed from scratch as standard input tokens.


### Task 2.4: With Caching

To enable prompt caching in Bedrock Converse API, define a cache checkpoint marker:
```python
cache_point = {"cachePoint": {"type": "default"}}
```
Placing `cache_point` after the `system_prompt` marks the preceding text as a cacheable prefix.

> [!IMPORTANT]
> **Understanding Cache Write vs. Cache Read**:
> - **Call 1 (Cache Write)**: The first call populates the cache (`cacheWriteInputTokens: 87`, `cacheReadInputTokens: 0`).
> - **Call 2 (Cache Read)**: Subsequent calls with the same prefix read directly from the cache (`cacheReadInputTokens: 87`, `cacheWriteInputTokens: 0`). Freshly computed tokens drop down to just `inputTokens + outputTokens` = **~46 tokens**!


In [4]:
cache_point = {"cachePoint": {"type": "default"}}

payload_with_caching = {
    "system": [
        {"text": system_prompt},
        cache_point
    ],
    "messages": [
        {
            "role": "user",
            "content": [
                {"text": user_input_review + "\nSummary:"}
            ]
        }
    ]
}

# Run the cached request
response = bedrock.converse(
    modelId=MODEL_ID,
    **payload_with_caching
)

output_text = response['output']['message']['content'][0]['text']
usage = response['usage']

cache_read = usage.get('cacheReadInputTokens', 0)
cache_write = usage.get('cacheWriteInputTokens', 0)
freshly_billed_tokens = usage['inputTokens'] + usage['outputTokens']

print("[With Caching] Generated Summary:")
print(output_text)
print(f"\nToken Usage Details:")
print(f"  Freshly Computed Tokens (input + output): {freshly_billed_tokens}")
print(f"  Cache Read Tokens:                        {cache_read}")
print(f"  Cache Write Tokens:                       {cache_write}")
print(f"  Total Context Window (raw tokens):        {usage['totalTokens']}")

if cache_read > 0:
    print(f"\n✅ Cache HIT! {cache_read} tokens were read from cache instead of being recomputed.")
    print(f"   Fresh tokens reduced from {total_tokens_no_cache} down to {freshly_billed_tokens} (saved {cache_read} tokens, ~65% reduction)!")
else:
    print(f"\n📝 First Call = Cache WRITE ({cache_write} tokens written to cache).")
    print("   Run this cell a 2nd time (or run the cell below) to see the Cache READ in action!")


[With Caching] Generated Summary:
Reviewer appreciates the new EP as a solid effort, highlighting a few standout songs while mentioning that some tracks feel repetitive.

Token Usage Details:
  Freshly Computed Tokens (input + output): 47
  Cache Read Tokens:                        87
  Cache Write Tokens:                       0
  Total Context Window (raw tokens):        134

✅ Cache HIT! 87 tokens were read from cache instead of being recomputed.
   Fresh tokens reduced from 132 down to 47 (saved 87 tokens, ~65% reduction)!


In [5]:
# Immediate Demonstration: Second Call (Cache READ)
# Using a different user review with the same cached system prompt!
second_review = "Review: The production quality on this track is stellar, but the lyrics feel uninspired."

second_payload = {
    "system": [
        {"text": system_prompt},
        cache_point
    ],
    "messages": [
        {
            "role": "user",
            "content": [
                {"text": second_review + "\nSummary:"}
            ]
        }
    ]
}

response2 = bedrock.converse(
    modelId=MODEL_ID,
    **second_payload
)

usage2 = response2['usage']
print("[Second Call with Cached System Prompt]")
print(response2['output']['message']['content'][0]['text'])
print(f"\nToken Usage Breakdown:")
print(f"  Cache Read Tokens:   {usage2.get('cacheReadInputTokens', 0)} (Served from cache!)")
print(f"  Cache Write Tokens:  {usage2.get('cacheWriteInputTokens', 0)}")
print(f"  Uncached Tokens:     {usage2['inputTokens']} input + {usage2['outputTokens']} output = {usage2['inputTokens'] + usage2['outputTokens']} tokens")
print(f"  Total Raw Context:   {usage2['totalTokens']}")


[Second Call with Cached System Prompt]
Reviewer appreciates the high production quality of the track but finds the lyrics to be uninspired.

Token Usage Breakdown:
  Cache Read Tokens:   0 (Served from cache!)
  Cache Write Tokens:  88
  Uncached Tokens:     18 input + 20 output = 38 tokens
  Total Raw Context:   126


> **Key Takeaway**:
> - In `response['usage']`, Bedrock reports `totalTokens` as the full context size.
> - But notice `cacheReadInputTokens: 87`: those 87 system prompt tokens were read from cache rather than reprocessed, reducing fresh compute from **~132 tokens down to ~46 tokens**!


## Task 3: Changing Your Bedrock Prompt

In this task, you'll modify your Bedrock instructions to observe how caching behaves when you change your original request.


In [6]:
# Updated system prompt for pet identification
system_prompt = """
You are a pet expert that will tell the user what type of pet they have based on a description. You'll keep your responses short and generalized.
Here are examples:

Description: My pet barks and has four legs.
Summary: Your pet is a dog.

Review: My pet meows and uses a litter box.
Summary: Your pet is a cat.
"""

# Input description for pet classification
pet_input = "Description: My pet has feathers, chirps, and lives in a cage."

print("Updated pet expert prompt and test input defined.")


Updated pet expert prompt and test input defined.


### Task 3.1: Execute with Caching Point
Run the updated prompt with `cache_point` included. We run it once to write to cache, then a second time to read from cache.


In [7]:
cache_point = {"cachePoint": {"type": "default"}}

payload_with_caching = {
    "system": [
        {"text": system_prompt},
        cache_point
    ],
    "messages": [
        {
            "role": "user",
            "content": [
                {"text": pet_input + "\nSummary:"}
            ]
        }
    ]
}

# Run request
response_pet = bedrock.converse(
    modelId=MODEL_ID,
    **payload_with_caching
)

u_pet = response_pet['usage']
print("[With Caching] Generated Summary:")
print(response_pet['output']['message']['content'][0]['text'])
print(f"\nToken Usage:")
print(f"  Freshly Computed Tokens: {u_pet['inputTokens'] + u_pet['outputTokens']}")
print(f"  Cache Read Tokens:       {u_pet.get('cacheReadInputTokens', 0)}")
print(f"  Cache Write Tokens:      {u_pet.get('cacheWriteInputTokens', 0)}")
print(f"  Total Raw Context:       {u_pet['totalTokens']}")


[With Caching] Generated Summary:
Your pet is a bird.

Token Usage:
  Freshly Computed Tokens: 24
  Cache Read Tokens:       0
  Cache Write Tokens:      75
  Total Raw Context:       99


### Task 3.2: Execute without Caching
Run the same updated prompt without caching to compare token usage.


In [8]:
no_cache_payload = {
    "system": [
        {"text": system_prompt}
    ],
    "messages": [
        {
            "role": "user",
            "content": [
                {"text": pet_input + "\nSummary:"}
            ]
        }
    ]
}

response_pet_no_cache = bedrock.converse(
    modelId=MODEL_ID,
    **no_cache_payload
)

u_pet_no_cache = response_pet_no_cache['usage']
print("[No Caching] Generated Summary:")
print(response_pet_no_cache['output']['message']['content'][0]['text'])
print(f"\nToken Usage (No Cache):")
print(f"  Input Tokens:  {u_pet_no_cache['inputTokens']}")
print(f"  Output Tokens: {u_pet_no_cache['outputTokens']}")
print(f"  Total Tokens:  {u_pet_no_cache['totalTokens']}")


[No Caching] Generated Summary:
Your pet is a bird.

Token Usage (No Cache):
  Input Tokens:  92
  Output Tokens: 7
  Total Tokens:  99


> **Comparison**:
> - **Without Caching**: Full prompt processed from scratch (~98 total tokens).
> - **With Caching**: Once written, fresh token computation drops to **~26 tokens** (`inputTokens + outputTokens`) because the ~72 system tokens are retrieved from cache!


## Task 4: Cleanup

- **Jupyter Server**: When you are finished, shut down your Jupyter server from the terminal.
- **Environment**: If you no longer need the Python virtual environment, deactivate it and delete the `exercise7` folder.


## Summary

You now know how to:
- ✅ **Use region `us-east-2`** with cross-region inference profiles (`us.amazon.nova-micro-v1:0`).
- ✅ **Understand the two stages of prompt caching**:
  1. **Cache Write**: Initial execution stores prefix tokens (`cacheWriteInputTokens`).
  2. **Cache Read**: Subsequent calls retrieve prefix tokens (`cacheReadInputTokens`), reducing freshly computed tokens from ~132 down to ~46.
- ✅ **Interpret Bedrock token usage metrics** (`totalTokens` vs. `cacheReadInputTokens` vs. fresh `inputTokens + outputTokens`).

---

### Additional Resources
- [Amazon Bedrock Documentation: Prompt Caching](https://docs.aws.amazon.com/bedrock/latest/userguide/prompt-caching.html)
- [Amazon Bedrock Converse API Reference](https://docs.aws.amazon.com/bedrock/latest/APIReference/API_runtime_Converse.html)
